In [ ]:
import tensorflow as tf
import keras

print(tf.__version__)
print(keras.__version__)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adam

file = "Datasets/train/images_datasets"
class_names = ['airplanes', 'buses', 'cars', 'motorcycles', 'ship']
num_classes = len(class_names)
image_generator = keras.preprocessing.image.ImageDataGenerator(validation_split=0.2,
                                                                  horizontal_flip=True,
                                                                  rotation_range=20,
                                                                  width_shift_range=0.2,
                                                                  height_shift_range=0.2,
                                                                  zoom_range=0.2,
                                                                  shear_range=0.2,
                                                                  fill_mode='nearest'
)
train_data_gen=image_generator.flow_from_directory(directory=file,
                                                   target_size=(256,256),
                                                   batch_size=12,
                                                   class_mode='categorical',
                                                   classes=class_names,
                                                   subset='training')
val_data_gen=image_generator.flow_from_directory(directory=file,
                                                 target_size=(256,256),
                                                 batch_size=12,
                                                 class_mode='categorical',
                                                 classes=class_names,
                                                 subset='validation')



In [ ]:
model = keras.models.Sequential([
    keras.Input(shape=(256, 256, 3)),
    keras.layers.Conv2D(32, (3, 3), strides=(1, 1), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(64, (3, 3), strides=(1, 1), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(128, (3, 3), strides=(1, 1), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(num_classes, activation='softmax')
])
adam_optimizer = Adam(learning_rate=0.0001)
model.compile(optimizer=adam_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
#early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history=model.fit(train_data_gen, validation_data=val_data_gen,  epochs=20, verbose=2)

In [ ]:
model.save('model.h5')
print("Model saved as 'model.h5'")

# Get class labels from the training generator
class_labels = list(train_data_gen.class_indices.keys())
print("Class labels:", class_labels)

In [ ]:
import numpy as np
from tensorflow import keras
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import os

test_dir = "Datasets/test/"


model_path = "model.h5"

img_height = 256
img_width = 256
batch_size = 32

try:
    model = keras.models.load_model(model_path, compile=False)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

# Create a data generator for the test data
test_datagen = keras.preprocessing.image.ImageDataGenerator()
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    classes=class_names,
    shuffle=False
)

true_classes = test_generator.classes
class_labels = class_names

# Make predictions
predictions = model.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)

# --- Calculate Metrics ---
accuracy = accuracy_score(true_classes, predicted_classes)
precision, recall, f1, support = precision_recall_fscore_support(true_classes, predicted_classes, average=None, labels=range(len(class_labels)))

print("\nPer-class metrics:")
print(f"{'Class':<12} {'Prec%':<8} {'Rec%':<8} {'F1%':<8} {'Support'}")
print("-" * 50)

for i, label in enumerate(class_labels):
    print(f"{label:<12} {precision[i]*100:<8.2f} {recall[i]*100:<8.2f} {f1[i]*100:<8.2f} {support[i]}")

print("\nMacro:")
print(f"Accuracy:         {accuracy*100:.2f}%")
print(f"Precision(macro): {np.mean(precision)*100:.2f}%")
print(f"Recall(macro):    {np.mean(recall)*100:.2f}%")
print(f"F1(macro):        {np.mean(f1)*100:.2f}%")

In [ ]:
import numpy as np
from tensorflow import keras

#img_path = "Testcases/test/airplanes/airplane1.jpg"

#img_path = "Testcases/test/cars/cars1.jpg"

img_path = "Datasets/test/airplanes/airplane9.jpg"

#img_path = "Testcases/test/airplanes/airplane1.jpg"

img = keras.preprocessing.image.load_img(img_path, target_size=(256, 256))
img_array = keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

predictions = model.predict(img_array)
predicted_class = np.argmax(predictions[0])
confidence = predictions[0][predicted_class]

print(f"Predicted class: {class_labels[predicted_class]}")
print(f"Confidence: {confidence:.4f}")

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import os

def test_single_image(img_path, model_path="model.h5"):
    """
    Test a single image with the trained CNN model and display results

    Args:
        img_path (str): Path to the image file
        model_path (str): Path to the saved model file
    """

    # Class labels from your model
    class_labels = class_names

    # Check if files exist
    if not os.path.exists(img_path):
        print(f"Image file not found: {img_path}")
        return

    if not os.path.exists(model_path):
        print(f"Model file not found: {model_path}")
        return

    # Load the trained model
    try:
        model = keras.models.load_model(model_path, compile=False)
        print("Model loaded successfully!")
    except Exception as e:
        print(f"Error loading model: {e}")
        return

    # Load and preprocess the image
    try:
        img = keras.preprocessing.image.load_img(img_path, target_size=(256, 256))
        img_array = keras.preprocessing.image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

        print(f"Image loaded and preprocessed: {img_path}")
    except Exception as e:
        print(f"Error processing image: {e}")
        return

    # Make prediction
    predictions = model.predict(img_array, verbose=0)
    predicted_class_idx = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class_idx]
    predicted_class = class_labels[predicted_class_idx]
    probabilities = predictions[0]

    plt.figure(figsize=(15, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f'Input Image\n\nPredicted: {predicted_class}\nConfidence: {confidence:.4f} ({confidence*100:.2f}%)',
             fontsize=14, fontweight='bold')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    colors = ['red' if i == predicted_class_idx else 'skyblue' for i in range(len(class_labels))]
    bars = plt.bar(class_labels, probabilities * 100, color=colors)
    plt.title('Class Probabilities', fontsize=14, fontweight='bold')
    plt.ylabel('Probability (%)', fontsize=12)
    plt.ylim(0, 100)

    for bar, prob in zip(bars, probabilities):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{prob*100:.1f}%', ha='center', va='bottom', fontweight='bold')

    plt.xticks(rotation=45, ha='right')

    plt.tight_layout()
    plt.show()

    print(f"\n{'='*50}")
    print(f"PREDICTION RESULTS")
    print(f"{'='*50}")
    print(f"Image: {os.path.basename(img_path)}")
    print(f"Predicted Class: {predicted_class}")
    print(f"Confidence: {confidence:.4f} ({confidence*100:.2f}%)")
    print(f"\nAll Class Probabilities:")
    print("-" * 30)
    for i, (label, prob) in enumerate(zip(class_labels, probabilities)):
        marker = "👉" if i == predicted_class_idx else "  "
        print(f"{marker} {label:<10}: {prob:.4f} ({prob*100:.2f}%)")
    print(f"{'='*50}")



if __name__ == "__main__":

    image_path = "Datasets/test/ship/2136335.jpg"


    test_single_image(image_path)

# Quick function call - uncomment and change path as needed:
# test_single_image("your_image_path_here.jpg")